# Universidad del Valle de Guatemala
# Inteligencia Artificial CC3085
# Proyecto 1
## Opción elegida: (A) Predicción de Tarifas de Transporte en NYC
## Integrantes del grupo:
### - Joel Jaquez #23369
### - Luis Gonzalez #23353
### - Diego Patzan #23525
### Fecha de entrega: Lunes de 6 de abril 2026

# Importación de librerias

In [ ]:
# Importación de librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración general de visualizaciones
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.float_format', '{:.4f}'.format)

print("Librerías cargadas correctamente")

# Carga del dataset

In [ ]:
# Carga del dataset
SAMPLE_SIZE = 1_000_000

df = pd.read_csv(
    '../data/train.csv',
    nrows=SAMPLE_SIZE,
    parse_dates=['pickup_datetime']
)

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")

# Fase 1: Análisis Exploratorio y Limpieza

In [ ]:
# Vista general
print("================")
print("PRIMERAS 5 FILAS")
print("================")
display(df.head())

print("================")
print("TIPOS DE DATOS")
print("================")
print(df.dtypes)

print("=========================")
print("ESTADÍSTICAS DESCRIPTIVAS")
print("=========================")
display(df.describe())

print("=============")
print("VALORES NULOS")
print("=============")
nulos = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
print(nulos)

## Análisis Exploratorio y Limpieza

In [ ]:
# Análisis Exploratorio: Distribuciones iniciales

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribuciones ANTES de Limpieza', fontsize=16, fontweight='bold')

axes[0,0].hist(df['fare_amount'], bins=100, color='steelblue', edgecolor='black')
axes[0,0].set_title('fare_amount')
axes[0,0].set_xlabel('USD')

axes[0,1].hist(df['pickup_longitude'], bins=100, color='coral', edgecolor='black')
axes[0,1].set_title('pickup_longitude')

axes[0,2].hist(df['pickup_latitude'], bins=100, color='coral', edgecolor='black')
axes[0,2].set_title('pickup_latitude')

axes[1,0].hist(df['dropoff_longitude'], bins=100, color='mediumpurple', edgecolor='black')
axes[1,0].set_title('dropoff_longitude')

axes[1,1].hist(df['dropoff_latitude'], bins=100, color='mediumpurple', edgecolor='black')
axes[1,1].set_title('dropoff_latitude')

axes[1,2].hist(df['passenger_count'], bins=20, color='mediumseagreen', edgecolor='black')
axes[1,2].set_title('passenger_count')

plt.tight_layout()
plt.savefig('distribucion_antes_limpieza.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafica guardada")

In [ ]:
# Análisis de Outliers (antes de limpiar)

print("============================================")
print("ANALISIS DE OUTLIERS - VALORES PROBLEMATICOS")
print("============================================")

print(f"\nfare_amount negativos o cero:  {(df['fare_amount'] <= 0).sum():,}")
print(f"fare_amount > $200:            {(df['fare_amount'] > 200).sum():,}")

print(f"\nCoordenadas fuera de NYC:")
LAT_MIN, LAT_MAX = 40.4774, 41.0755
LON_MIN, LON_MAX = -74.2591, -71.6966

fuera_pickup = (
    ~df['pickup_latitude'].between(LAT_MIN, LAT_MAX) |
    ~df['pickup_longitude'].between(LON_MIN, LON_MAX)
)
fuera_dropoff = (
    ~df['dropoff_latitude'].between(LAT_MIN, LAT_MAX) |
    ~df['dropoff_longitude'].between(LON_MIN, LON_MAX)
)
print(f"   - Pickup fuera de NYC:    {fuera_pickup.sum():,}")
print(f"   - Dropoff fuera de NYC:   {fuera_dropoff.sum():,}")

print(f"\npassenger_count = 0:          {(df['passenger_count'] == 0).sum():,}")
print(f"passenger_count > 7:          {(df['passenger_count'] > 7).sum():,}")

In [ ]:
# LIMPIEZA COMPLETA

df_clean = df.copy()
print(f"Filas iniciales: {len(df_clean):,}")

# 1. Eliminar nulos
df_clean = df_clean.dropna()
print(f"Tras eliminar nulos:           {len(df_clean):,}")

# 2. Limpieza de etiquetas (fare_amount)
df_clean = df_clean[df_clean['fare_amount'] > 0]
df_clean = df_clean[df_clean['fare_amount'] <= 200]
print(f"Tras limpiar fare_amount:      {len(df_clean):,}")

# 3. Limpieza geográfica (bounding box NYC)
df_clean = df_clean[
    df_clean['pickup_latitude'].between(LAT_MIN, LAT_MAX)  &
    df_clean['pickup_longitude'].between(LON_MIN, LON_MAX) &
    df_clean['dropoff_latitude'].between(LAT_MIN, LAT_MAX) &
    df_clean['dropoff_longitude'].between(LON_MIN, LON_MAX)
]
print(f"Tras limpieza geografica:      {len(df_clean):,}")

# 4. Limpieza de passenger_count
df_clean = df_clean[df_clean['passenger_count'].between(1, 7)]
print(f"Tras limpiar passenger_count:  {len(df_clean):,}")

# Resumen
eliminadas = 1_000_000 - len(df_clean)
print(f"\nFilas eliminadas: {eliminadas:,} ({eliminadas/1_000_000*100:.2f}%)")
print(f"Dataset limpio:   {len(df_clean):,} filas")

In [ ]:
# Distribuciones DESPUÉS de limpieza

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribuciones DESPUES de Limpieza', fontsize=16, fontweight='bold')

axes[0,0].hist(df_clean['fare_amount'], bins=80, color='steelblue', edgecolor='black')
axes[0,0].set_title('fare_amount')
axes[0,0].set_xlabel('USD')

axes[0,1].hist(df_clean['pickup_longitude'], bins=80, color='coral', edgecolor='black')
axes[0,1].set_title('pickup_longitude')

axes[0,2].hist(df_clean['pickup_latitude'], bins=80, color='coral', edgecolor='black')
axes[0,2].set_title('pickup_latitude')

axes[1,0].hist(df_clean['dropoff_longitude'], bins=80, color='mediumpurple', edgecolor='black')
axes[1,0].set_title('dropoff_longitude')

axes[1,1].hist(df_clean['dropoff_latitude'], bins=80, color='mediumpurple', edgecolor='black')
axes[1,1].set_title('dropoff_latitude')

axes[1,2].bar(
    df_clean['passenger_count'].value_counts().sort_index().index,
    df_clean['passenger_count'].value_counts().sort_index().values,
    color='mediumseagreen', edgecolor='black'
)
axes[1,2].set_title('passenger_count')
axes[1,2].set_xlabel('Numero de pasajeros')

plt.tight_layout()
plt.savefig('distribucion_despues_limpieza.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafica guardada")

In [ ]:
# Mapa de calor de correlaciones

cols_corr = ['fare_amount', 'pickup_longitude', 'pickup_latitude',
             'dropoff_longitude', 'dropoff_latitude', 'passenger_count']

plt.figure(figsize=(9, 7))
sns.heatmap(
    df_clean[cols_corr].corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5
)
plt.title('Matriz de Correlacion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlacion.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafica guardada")

# Fase 2: Ingeniería Característica

In [ ]:

# Distancia Haversine

#Calcula la distancia en km entre dos puntos geograficos usando la formula de Haversine.
def haversine(lat1, lon1, lat2, lon2):

    R = 6371  # Radio de la Tierra en km

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

# Aplicar Haversine al dataset limpio
df_clean['distance_km'] = haversine(
    df_clean['pickup_latitude'].values,
    df_clean['pickup_longitude'].values,
    df_clean['dropoff_latitude'].values,
    df_clean['dropoff_longitude'].values
)

print("Estadisticas de distancia_km:")
print(df_clean['distance_km'].describe())

In [ ]:
# Filtrar distancias imposibles (outliers restantes)

print(f"Filas antes de filtrar distancias: {len(df_clean):,}")

# Eliminar viajes con distancia 0 (pickup == dropoff) o mayores a 100 km
df_clean = df_clean[df_clean['distance_km'] > 0]
df_clean = df_clean[df_clean['distance_km'] <= 100]

print(f"Filas despues de filtrar distancias: {len(df_clean):,}")

In [ ]:
# Variables Temporales

# Extraer componentes del timestamp
df_clean['hour']       = df_clean['pickup_datetime'].dt.hour
df_clean['day_of_week'] = df_clean['pickup_datetime'].dt.dayofweek  # 0=Lunes, 6=Domingo
df_clean['month']      = df_clean['pickup_datetime'].dt.month
df_clean['year']       = df_clean['pickup_datetime'].dt.year

print("Variables temporales creadas:")
print(df_clean[['pickup_datetime', 'hour', 'day_of_week', 'month', 'year']].head(8))

In [ ]:
# Zonas de Interes (Aeropuertos)

# Coordenadas aproximadas de los aeropuertos principales de NYC
AIRPORTS = {
    'jfk':      {'lat': 40.6413, 'lon': -73.7781, 'radio': 0.05},
    'laguardia': {'lat': 40.7769, 'lon': -73.8740, 'radio': 0.03},
    'newark':   {'lat': 40.6895, 'lon': -74.1745, 'radio': 0.04},
}

def cerca_aeropuerto(lat, lon, aeropuerto):
    """Retorna 1 si el punto esta dentro del radio del aeropuerto."""
    a = AIRPORTS[aeropuerto]
    return ((np.abs(lat - a['lat']) < a['radio']) &
            (np.abs(lon - a['lon']) < a['radio'])).astype(int)

# Variable binaria: origen cerca de algun aeropuerto
df_clean['pickup_jfk']       = cerca_aeropuerto(df_clean['pickup_latitude'], df_clean['pickup_longitude'], 'jfk')
df_clean['pickup_laguardia'] = cerca_aeropuerto(df_clean['pickup_latitude'], df_clean['pickup_longitude'], 'laguardia')
df_clean['pickup_newark']    = cerca_aeropuerto(df_clean['pickup_latitude'], df_clean['pickup_longitude'], 'newark')

# Variable binaria: destino cerca de algun aeropuerto
df_clean['dropoff_jfk']       = cerca_aeropuerto(df_clean['dropoff_latitude'], df_clean['dropoff_longitude'], 'jfk')
df_clean['dropoff_laguardia'] = cerca_aeropuerto(df_clean['dropoff_latitude'], df_clean['dropoff_longitude'], 'laguardia')
df_clean['dropoff_newark']    = cerca_aeropuerto(df_clean['dropoff_latitude'], df_clean['dropoff_longitude'], 'newark')

airport_cols = ['pickup_jfk', 'pickup_laguardia', 'pickup_newark',
                'dropoff_jfk', 'dropoff_laguardia', 'dropoff_newark']

print("Viajes relacionados con aeropuertos:")
print(df_clean[airport_cols].sum())

In [ ]:
# Visualizacion del Feature Engineering

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Distribucion de Variables Nuevas', fontsize=14, fontweight='bold')

# Distancia
axes[0].hist(df_clean['distance_km'], bins=80, color='steelblue', edgecolor='black')
axes[0].set_title('Distancia (km)')
axes[0].set_xlabel('km')

# Hora del dia
axes[1].bar(
    df_clean['hour'].value_counts().sort_index().index,
    df_clean['hour'].value_counts().sort_index().values,
    color='coral', edgecolor='black'
)
axes[1].set_title('Hora del dia')
axes[1].set_xlabel('Hora')

# Dia de la semana
dias = ['Lun', 'Mar', 'Mie', 'Jue', 'Vie', 'Sab', 'Dom']
axes[2].bar(
    df_clean['day_of_week'].value_counts().sort_index().index,
    df_clean['day_of_week'].value_counts().sort_index().values,
    color='mediumseagreen', edgecolor='black'
)
axes[2].set_xticks(range(7))
axes[2].set_xticklabels(dias)
axes[2].set_title('Dia de la semana')

plt.tight_layout()
plt.savefig('feature_engineering.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafica guardada")

In [ ]:
#  Resumen final del dataset con todas las features

print(f"Shape final del dataset: {df_clean.shape}")
print(f"\nColumnas disponibles:")
print(list(df_clean.columns))
print(f"\nPrimeras filas con todas las features:")
display(df_clean.head())

# Fase 3: Partición Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

# Features (variables de entrada)
X = df_clean[['distance_km', 'hour', 'day_of_week', 'month', 'year',
              'passenger_count', 'pickup_longitude', 'pickup_latitude',
              'dropoff_longitude', 'dropoff_latitude',
              'pickup_jfk', 'pickup_laguardia', 'pickup_newark',
              'dropoff_jfk', 'dropoff_laguardia', 'dropoff_newark']]

# Target (variable a predecir)
y = df_clean['fare_amount']

# División 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42  # garantiza reproducibilidad
)

print("Particion completada:")
print(f"  X_train: {X_train.shape[0]:,} filas x {X_train.shape[1]} columnas")
print(f"  X_test:  {X_test.shape[0]:,} filas x {X_test.shape[1]} columnas")
print(f"  y_train: {y_train.shape[0]:,} valores")
print(f"  y_test:  {y_test.shape[0]:,} valores")
print(f"\n  Train: {X_train.shape[0]/len(X)*100:.1f}%")
print(f"  Test:  {X_test.shape[0]/len(X)*100:.1f}%")

# Fase 4: Experimentación con modelos

## Modelo de Regresión Lineal

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

# Entrenamiento 
print("Entrenando Regresion Lineal...")
start = time.time()

modelo_lr = LinearRegression()
modelo_lr.fit(X_train, y_train)

tiempo_entrenamiento = time.time() - start
print(f"Tiempo de entrenamiento: {tiempo_entrenamiento:.2f} segundos")

In [ ]:
# Predicciones

y_pred_train_lr = modelo_lr.predict(X_train)
y_pred_test_lr  = modelo_lr.predict(X_test)

In [ ]:
# Métricas

def calcular_metricas(y_real, y_pred, nombre):
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2   = r2_score(y_real, y_pred)
    print(f"\n--- {nombre} ---")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R2:   {r2:.4f}")
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

print("===========================")
print("METRICAS - REGRESION LINEAL")
print("===========================")

metricas_train_lr = calcular_metricas(y_train, y_pred_train_lr, "Train")
metricas_test_lr  = calcular_metricas(y_test,  y_pred_test_lr,  "Test")

In [ ]:
# Coeficientes del modelo

print("\n--- Coeficientes por variable ---")
coeficientes = pd.DataFrame({
    'Feature': X_train.columns,
    'Coeficiente': modelo_lr.coef_
}).sort_values('Coeficiente', ascending=False)
print(coeficientes.to_string(index=False))
print(f"\nIntercepto: {modelo_lr.intercept_:.4f}")

In [ ]:
# ── Gráfica Predicciones vs Valores Reales ───────────────────

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_test_lr, alpha=0.05, s=1, color='steelblue')
plt.plot([0, 200], [0, 200], color='red', linewidth=1.5, label='Prediccion perfecta')
plt.xlabel('Tarifa Real (USD)')
plt.ylabel('Tarifa Predicha (USD)')
plt.title('Regresion Lineal: Predicciones vs Valores Reales')
plt.legend()
plt.tight_layout()
plt.savefig('lr_predicciones_vs_reales.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafica guardada")

In [ ]:
# Gráfica de Residuos

residuos = y_test.values - y_pred_test_lr

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_test_lr, residuos, alpha=0.05, s=1, color='coral')
plt.axhline(y=0, color='black', linewidth=1.5)
plt.xlabel('Tarifa Predicha (USD)')
plt.ylabel('Residuo (Real - Predicho)')
plt.title('Regresion Lineal: Grafica de Residuos')
plt.tight_layout()
plt.savefig('lr_residuos.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafica guardada")

## Modelo 2: Random Forest


## Busqueda de hiperparametros

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import time
features = list(X_train.columns)

# Submuestra para grid search (con 770k filas tarda mucho)
SAMPLE_GS = 100_000
idx_sample = X_train.sample(n=SAMPLE_GS, random_state=42).index
X_sample = X_train.loc[idx_sample]
y_sample = y_train.loc[idx_sample]

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [15, 20, None],
    "min_samples_split": [5, 10]
}

print("Buscando mejores hiperparametros con GridSearchCV (3-fold)...")
print(f"Usando submuestra de {SAMPLE_GS:,} filas")
start = time.time()

gs = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid, cv=3,
    scoring="neg_mean_absolute_error",
    verbose=1, n_jobs=-1
)
gs.fit(X_sample, y_sample)

tiempo_gs = time.time() - start
print(f"\nGrid Search terminado en {tiempo_gs:.1f} segundos")
print(f"Mejor combinacion: {gs.best_params_}")
print(f"Mejor MAE (CV): {-gs.best_score_:.4f}")

In [ ]:
# Resultados completos del grid search
resultados_gs = pd.DataFrame(gs.cv_results_)
cols = ["param_n_estimators", "param_max_depth", "param_min_samples_split",
        "mean_test_score", "std_test_score", "rank_test_score"]
resultados_gs = resultados_gs[cols].sort_values("rank_test_score")
resultados_gs["mean_test_score"] = -resultados_gs["mean_test_score"]
resultados_gs.columns = ["n_estimators", "max_depth", "min_samples_split", "MAE", "std", "rank"]
print("Todas las combinaciones probadas:")
display(resultados_gs)

## Entrenamiento con dataset completo

In [ ]:
print("Entrenando Random Forest con todos los datos de train...")
start = time.time()

modelo_rf = RandomForestRegressor(
    **gs.best_params_,
    random_state=42,
    n_jobs=-1
)
modelo_rf.fit(X_train, y_train)

tiempo_rf = time.time() - start
print(f"Tiempo de entrenamiento: {tiempo_rf:.1f} segundos")
print(f"Hiperparametros: {gs.best_params_}")

In [ ]:

def calcular_metricas(y_real, y_pred, nombre):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    import numpy as np
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2   = r2_score(y_real, y_pred)
    print(f"\n--- {nombre} ---")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R2:   {r2:.4f}")
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

# Predicciones y metricas
y_pred_train_rf = modelo_rf.predict(X_train)
y_pred_test_rf = modelo_rf.predict(X_test)

print("===========================")
print("METRICAS - RANDOM FOREST")
print("===========================")

metricas_train_rf = calcular_metricas(y_train, y_pred_train_rf, "Train")
metricas_test_rf = calcular_metricas(y_test, y_pred_test_rf, "Test")

## Graficas de resultados

In [ ]:
# Predicciones vs Valores Reales
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_test_rf, alpha=0.05, s=1, color="forestgreen")
plt.plot([0, 200], [0, 200], color="red", linewidth=1.5, label="Prediccion perfecta")
plt.xlabel("Tarifa Real (USD)")
plt.ylabel("Tarifa Predicha (USD)")
plt.title("Random Forest: Predicciones vs Valores Reales")
plt.legend()
plt.tight_layout()
plt.savefig("rf_predicciones_vs_reales.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafica guardada")

In [ ]:
# Residuos
residuos_rf = y_test.values - y_pred_test_rf

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_test_rf, residuos_rf, alpha=0.05, s=1, color="darkorange")
plt.axhline(y=0, color="black", linewidth=1.5)
plt.xlabel("Tarifa Predicha (USD)")
plt.ylabel("Residuo (Real - Predicho)")
plt.title("Random Forest: Grafica de Residuos")
plt.tight_layout()
plt.savefig("rf_residuos.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafica guardada")

In [ ]:
# Importancia de features
importancias = pd.DataFrame({
    "Feature": features,
    "Importancia": modelo_rf.feature_importances_
}).sort_values("Importancia", ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(importancias["Feature"], importancias["Importancia"], color="teal")
plt.xlabel("Importancia")
plt.title("Random Forest: Importancia de Variables")
plt.tight_layout()
plt.savefig("rf_importancia_features.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafica guardada")

## Analisis de puntos de alta influencia

In [ ]:
# Top 10 predicciones con mayor error
errores = np.abs(y_test.values - y_pred_test_rf)
idx_peores = np.argsort(errores)[-10:][::-1]

df_peores = X_test.iloc[idx_peores].copy()
df_peores["tarifa_real"] = y_test.values[idx_peores]
df_peores["prediccion"] = y_pred_test_rf[idx_peores]
df_peores["error_abs"] = errores[idx_peores]

print("Top 10 predicciones con mayor error:")
display(df_peores[["distance_km", "hour", "tarifa_real", "prediccion", "error_abs",
                    "pickup_jfk", "dropoff_jfk", "pickup_newark", "dropoff_newark"]])

In [ ]:
# Comparativa train vs test (detectar overfitting)
comparacion = pd.DataFrame({
    "Metrica": ["MAE", "RMSE", "R2"],
    "Train": [metricas_train_rf["MAE"], metricas_train_rf["RMSE"], metricas_train_rf["R2"]],
    "Test": [metricas_test_rf["MAE"], metricas_test_rf["RMSE"], metricas_test_rf["R2"]]
})
comparacion["Diferencia"] = comparacion["Train"] - comparacion["Test"]

print("Comparacion Train vs Test:")
display(comparacion)

diff_r2 = metricas_train_rf["R2"] - metricas_test_rf["R2"]
if diff_r2 > 0.05:
    print(f"\nAtencion: diferencia de R2 = {diff_r2:.4f}, posible overfitting")
else:
    print(f"\nDiferencia de R2 = {diff_r2:.4f}, no hay overfitting significativo")

## Modelo 3 Redes Neuronales Densas (MLP)

In [ ]:
import os

MLP_DIR = "Redes Neuronales Densas"
os.makedirs(MLP_DIR, exist_ok=True)
print(f"Carpeta '{MLP_DIR}/' creada correctamente")

In [ ]:
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import time
import warnings
warnings.filterwarnings('ignore')

SEED = 42
tf.random.set_seed(SEED)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("StandardScaler aplicado")
print(f"X_train_sc shape: {X_train_sc.shape}")
print(f"X_test_sc  shape: {X_test_sc.shape}")